[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# Dates and Times &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import math
from datetime import datetime, timedelta

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

print("pandas", pd.__version__, "|", len(readings), "readings,", len(stations), "stations")

pandas 2.2.3 | 35040 readings, 5 stations


**1.** Parse the times `"31/12/2025 23:00"` and `"01/01/2026 00:00"`, which are written day first, with an explicit `format`, and print how far apart they are by subtracting the first from the second.

In [2]:
times = pd.to_datetime(pd.Series(["31/12/2025 23:00", "01/01/2026 00:00"]), format="%d/%m/%Y %H:%M")

print(times.iloc[1] - times.iloc[0])

0 days 01:00:00


Subtracting two Timestamps gives a Timedelta, here one hour, across the change of year.

**2.** Count Tromso's readings taken from 12:00 to 14:00 UTC, both hours included, using `.dt.hour`.

In [3]:
tromso = readings[readings["station"] == "Tromso"]

print(tromso["time"].dt.hour.between(12, 14).sum())

1095


`between` includes both ends, so three hours a day for 365 days.

**3.** Resample Bergen's readings to weekly means with `"W"`, and print the warmest week and its mean.

In [4]:
bergen = readings[readings["station"] == "Bergen"].set_index("time")["celsius"]
weekly = bergen.resample("W").mean().round(2)

print(len(weekly), "weeks")
print(weekly.idxmax(), weekly.max())

53 weeks
2025-07-20 00:00:00 17.0


Each week is labeled by the Sunday it ends on, so the warmest week ended on 20 July. 2025 began on a Wednesday, so the first week holds five days, and 53 labels cover the year.

**4.** Make a series of Oslo's readings indexed by local time, and print the local days that do not have 24 readings, with `resample("D").size()`.

In [5]:
oslo = readings[readings["station"] == "Oslo"]
local = oslo["time"].dt.tz_localize("UTC").dt.tz_convert("Europe/Oslo")
by_local = pd.Series(oslo["celsius"].to_numpy(), index=local)

hours = by_local.resample("D").size()
hours[hours != 24]

time
2025-01-01 00:00:00+01:00    23
2025-03-30 00:00:00+01:00    23
2025-10-26 00:00:00+02:00    25
2026-01-01 00:00:00+01:00     1
dtype: int64

The four days are the two clock changes and both ends of the year, as the worked example found by counting dates.

**5.** Make a 7-day rolling mean of Oslo's daily means in UTC days, and print its lowest value and the day its window ends on.

In [6]:
oslo = readings[readings["station"] == "Oslo"].set_index("time")["celsius"]
weekly_mean = oslo.resample("D").mean().rolling(7).mean()

print(round(weekly_mean.min(), 2), weekly_mean.idxmin())

-2.51 2025-01-20 00:00:00


A rolling window is labeled by its last value, so the coldest seven days ended on 20 January.

---

&#8592; **Back to:** [Dates and Times](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/16-dates-and-times.ipynb)  &nbsp;&middot;&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)
